**المرحلة الاولى **

In [ ]:
# ============================================================
# المرحلة الأولى: إعداد البيانات
# ============================================================

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------------------------------------
# 1. تحميل البيانات
# -----------------------------------------------------------
# في Kaggle، المسار يكون كالتالي:
train_path = '/kaggle/input/competitions/digit-recognizer/train.csv'

# قراءة ملف CSV
data = pd.read_csv(train_path)

print(f"📊 شكل البيانات الكاملة: {data.shape}")
# المتوقع: (42000, 785) → 42000 صورة، 784 عمود بكسل + 1 عمود label

# -----------------------------------------------------------
# 2. فصل التسميات عن البكسلات
# -----------------------------------------------------------
# العمود 'label' يحتوي على الرقم (0 إلى 9)
labels = data['label']

# باقي الأعمدة هي قيم البكسلات (784 عمود)
pixels = data.drop('label', axis=1)

print(f"🔢 عدد التسميات: {len(labels)}")
print(f"🖼️  عدد البكسلات: {pixels.shape}")


In [ ]:
# -----------------------------------------------------------
# 3. التقسيم الثلاثي (Train / Validation / Test)
# -----------------------------------------------------------

# أولاً: التطبيع البسيط (قسمة على 255)
# هذا يضع القيم بين 0 و 1
X = pixels.values / 255.0
y = labels.values

# الخطوة 3أ: فصل مجموعة الاختبار (20% من الكل)
# stratify=y يضمن توازن الأرقام في كل مجموعة
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,       # 20% للاختبار
    random_state=42,      # ثبات العشوائية
    stratify=y            # توازن التوزيع للأرقام (0-9)
)

# الخطوة 3ب: فصل مجموعة التحقق من الباقي (80%)
# نريد 15% من الكل = 15/80 من الباقي
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1875,     # 15% ÷ 80% = 0.1875
    random_state=42,
    stratify=y_temp       # توازن التوزيع أيضاً
)

print(f"\n📦 التدريب: {len(X_train)} صورة ({len(X_train)/len(X)*100:.1f}%)")
print(f"📦 التحقق:   {len(X_val)} صورة ({len(X_val)/len(X)*100:.1f}%)")
print(f"📦 الاختبار: {len(X_test)} صورة ({len(X_test)/len(X)*100:.1f}%)")

# التحقق السريع من توازن الأرقام
print(f"\n🔢 توزيع الأرقام في التدريب:")
print(np.bincount(y_train))

# -----------------------------------------------------------
# 4. التسوية المعيارية (Standardization)
# -----------------------------------------------------------
# نحسب المتوسط والانحراف المعياري من بيانات التدريب فقط
train_mean = X_train.mean()
train_std = X_train.std()

print(f"\n📐 متوسط التدريب: {train_mean:.4f}")
print(f"📐 انحراف التدريب المعياري: {train_std:.4f}")

# نطبق على الثلاث مجموعات باستخدام إحصائيات التدريب
X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

# -----------------------------------------------------------
# 5. التحويل إلى Tensors (لغة PyTorch)
# -----------------------------------------------------------
# dtype=torch.float32 للبيانات (أرقام عشرية)
# dtype=torch.long للتسميات (أرقام صحيحة للتصنيف)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


In [ ]:
# -----------------------------------------------------------
# 6. إنشاء DataLoaders (للتدريب بالدفعات)
# -----------------------------------------------------------
# DataLoader يقسم البيانات إلى دفعات (Batches) صغيرة
# هذا يمنع استنزاف ذاكرة GPU ويُسرّع التدريب

batch_size = 64

# TensorDataset يربط البيانات بالتسميات في كائن واحد
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# DataLoader للتدريب: shuffle=True يخلط البيانات في كل دورة
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# DataLoader للتحقق والاختبار: لا حاجة للخلط
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\n✅ عدد دفعات التدريب: {len(train_loader)}")
print(f"✅ عدد دفعات التحقق: {len(val_loader)}")
print(f"✅ عدد دفعات الاختبار: {len(test_loader)}")

# -----------------------------------------------------------
# 7. التحقق من الجهاز (GPU أو CPU)
# -----------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  الجهاز المستخدم: {device}")

**المرحلة الثانية : بناء النموذج**

In [ ]:
# ============================================================
# المرحلة الثانية: بناء النموذج (CNN)
# ============================================================

import torch
import torch.nn as nn

# تحديد الجهاز
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DigitCNN(nn.Module):
    def __init__(self):
        super(DigitCNN, self).__init__()
        
        # -------------------------------------------------------
        # الكتلة الأولى: من 1 قناة (رمادي) إلى 32 قناة
        # الدخل: 28 × 28
        # -------------------------------------------------------
        self.block1 = nn.Sequential(
            # طبقة تلافيفية: 1 قناة → 32 قناة، مرشح 3×3
            # padding=1 يحافظ على الحجم (28×28)
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),   # تسوية دفعات للقنوات الـ 32
            nn.ReLU(),            # التنشيط
            
            # طبقة تلافيفية ثانية: 32 → 32
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            # تجميع أقصى: يقلص الصورة من 28×28 إلى 14×14
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # -------------------------------------------------------
        # الكتلة الثانية: من 32 قناة إلى 64 قناة
        # الدخل: 14 × 14
        # -------------------------------------------------------
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            # تجميع أقصى: يقلص من 14×14 إلى 7×7
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # -------------------------------------------------------
        # الطبقات الخطية (التصنيف)
        # -------------------------------------------------------
        # بعد كتلتين: 64 قناة × 7 × 7 = 3136 قيمة
        self.classifier = nn.Sequential(
            nn.Flatten(),                    # تسطيح: (64,7,7) → (3136)
            
            nn.Linear(64 * 7 * 7, 128),      # طبقة مخفية
            nn.BatchNorm1d(128),             # تسوية دفعات للبُعد 1
            nn.ReLU(),
            nn.Dropout(0.5),                 # إسقاط عشوائي 50%
            
            nn.Linear(128, 10)               # المخرج: 10 أرقام (0-9)
            # لا نضع SoftMax هنا لأن CrossEntropyLoss تتضمنه داخلياً
        )
    
    def forward(self, x):
        # x يأتي بشكل (batch_size, 784)
        # نُعيد تشكيله إلى (batch_size, 1, 28, 28)
        x = x.view(-1, 1, 28, 28)
        
        x = self.block1(x)   # الكتلة الأولى
        x = self.block2(x)   # الكتلة الثانية
        x = self.classifier(x)  # التصنيف
        
        return x

# إنشاء النموذج ونقله للجهاز
model = DigitCNN().to(device)

# طباعة ملخص للنموذج
print(model)

# حساب عدد المعاملات
total_params = sum(p.numel() for p in model.parameters())
print(f"\n🔢 إجمالي المعاملات: {total_params:,}")

**** المرحلة الثالثة اعداد اليات التدريب****

In [ ]:
# ============================================================
# المرحلة الثالثة: إعداد آليات التدريب
# ============================================================

import torch.nn as nn
import torch.optim as optim

# -----------------------------------------------------------
# 1. دالة الخسارة
# -----------------------------------------------------------
criterion = nn.CrossEntropyLoss()

# -----------------------------------------------------------
# 2. المُحسّن (Adam)
# -----------------------------------------------------------
# lr=0.001 هو الخيار الافتراضي الآمن
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -----------------------------------------------------------
# 3. مُخفض معدل التعلم
# -----------------------------------------------------------
# يراقب دقة التحقق (mode='max')
# إذا لم تتحسن لـ 3 دورات → اقسم lr على 10
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.1,
    patience=3,
 
)

# -----------------------------------------------------------
# 4. متغيرات الإيقاف المبكر
# -----------------------------------------------------------
best_val_acc = 0.0
patience_counter = 0
patience = 7
best_model_weights = None

# -----------------------------------------------------------
# 5. إعداد DataLoaders (من المرحلة الأولى)
# -----------------------------------------------------------
# نحتاج لإعادة تشكيل البيانات لأن CNN تتوقع صوراً
# البيانات حالياً (N, 784)، نريد (N, 1, 28, 28)

# إعادة التشكيل للتدريب
X_train_cnn = X_train_tensor.view(-1, 1, 28, 28)
X_val_cnn = X_val_tensor.view(-1, 1, 28, 28)
X_test_cnn = X_test_tensor.view(-1, 1, 28, 28)

# إنشاء DataLoaders جديدة بالأشكال الصحيحة
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_cnn, y_train_tensor)
val_dataset = TensorDataset(X_val_cnn, y_val_tensor)
test_dataset = TensorDataset(X_test_cnn, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("✅ جميع آليات التدريب جاهزة.")
print(f"   عدد دفعات التدريب: {len(train_loader)}")
print(f"   عدد دفعات التحقق: {len(val_loader)}")

**المرحلة الرابعة التدريب**

In [ ]:
# ============================================================
# المرحلة الرابعة: حلقة التدريب
# ============================================================

num_epochs = 50  # حد أقصى. الإيقاف المبكر قد يوقفنا قبلها

print("🚀 بدء التدريب...\n")

for epoch in range(num_epochs):
    # -------------------------------------------------------
    # الجزء الأول: التدريب (Training)
    # -------------------------------------------------------
    model.train()  # وضع التدريب: يُفعل Dropout و BatchNorm
    
    running_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        # نقل البيانات للجهاز (GPU/CPU)
        images = images.to(device)
        labels = labels.to(device)
        
        # 1. تصفير الميلانات
        optimizer.zero_grad()
        
        # 2. التمرير الأمامي
        outputs = model(images)
        
        # 3. حساب الخسارة
        loss = criterion(outputs, labels)
        
        # 4. الانتشار العكسي
        loss.backward()
        
        # 5. تحديث الأوزان
        optimizer.step()
        
        # إحصائيات
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    
    train_acc = 100 * train_correct / train_total
    avg_train_loss = running_loss / len(train_loader)
    
    # -------------------------------------------------------
    # الجزء الثاني: التحقق (Validation)
    # -------------------------------------------------------
    model.eval()  # وضع التقييم: يُعطل Dropout
    
    val_correct = 0
    val_total = 0
    val_loss = 0.0
    
    with torch.no_grad():  # لا نحسب الميلانات في التحقق (يوفر وقت وذاكرة)
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)
    
    # -------------------------------------------------------
    # الجزء الثالث: الطباعة والقرارات
    # -------------------------------------------------------
    print(f"📊 الدورة [{epoch+1:02d}/{num_epochs}] | "
          f"خسارة التدريب: {avg_train_loss:.4f} | "
          f"دقة التدريب: {train_acc:.2f}% | "
          f"خسارة التحقق: {avg_val_loss:.4f} | "
          f"دقة التحقق: {val_acc:.2f}%")
    
    # تحديث جدولة معدل التعلم
    scheduler.step(val_acc)
    
    # الإيقاف المبكر + حفظ أفضل أوزان
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        best_model_weights = model.state_dict().copy()
        print(f"   💾 تم حفظ أفضل أوزان (دقة: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"   ⏳ لم تتحسن الدقة ({patience_counter}/{patience})")
        
        if patience_counter >= patience:
            print(f"\n🛑 الإيقاف المبكر! أفضل دقة: {best_val_acc:.2f}%")
            break

# -------------------------------------------------------
# استرجاع أفضل أوزان
# -------------------------------------------------------
if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
    print(f"\n✅ تم استرجاع أفضل أوزان بدقة تحقق: {best_val_acc:.2f}%")

**المرحلة الخامسة الاختبار**

In [ ]:
# ============================================================
# المرحلة الخامسة: التقييم النهائي على مجموعة الاختبار
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# -----------------------------------------------------------
# 1. التقييم على مجموعة الاختبار (مرة واحدة فقط!)
# -----------------------------------------------------------
model.eval()  # وضع التقييم

test_correct = 0
test_total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        # حفظ جميع التوقعات والحقائق لتحليل لاحق
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = 100 * test_correct / test_total
print(f"🎯 دقة الاختبار النهائية: {test_acc:.2f}%")
print(f"   صحيحة: {test_correct} من أصل {test_total}")

# -----------------------------------------------------------
# 2. مصفوفة الالتباس (Confusion Matrix)
# -----------------------------------------------------------
# تُظهر لنا: أي رقمين يُخطئ النموذج في التمييز بينهما؟

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('التوقع (Predicted)')
plt.ylabel('الحقيقي (Actual)')
plt.title('مصفوفة الالتباس: أين يخطئ النموذج؟')
plt.show()

# -----------------------------------------------------------
# 3. التقرير التفصيلي (Classification Report)
# -----------------------------------------------------------
# يُظهر: الدقة، الاسترجاع، وF1-Score لكل رقم على حدة

print("\n📋 التقرير التفصيلي:")
print(classification_report(all_labels, all_preds, digits=4))

**الاختبار على ملف الاختبار**

In [ ]:
import pandas as pd
import torch

# 1. تحميل بيانات Kaggle Test
test_df = pd.read_csv('/kaggle/input/competitions/digit-recognizer/test.csv')

# 2. تطبيق نفس التطبيع بالضبط (إحصائيات التدريب فقط)
X_kaggle = test_df.values / 255.0
X_kaggle = (X_kaggle - train_mean) / train_std

# 3. إعادة التشكيل إلى صور
X_kaggle_tensor = torch.tensor(X_kaggle, dtype=torch.float32).view(-1, 1, 28, 28)

# 4. التوقع
model.eval()
kaggle_preds = []

with torch.no_grad():
    for i in range(len(X_kaggle_tensor)):
        image = X_kaggle_tensor[i].unsqueeze(0).to(device)
        output = model(image)
        _, pred = torch.max(output, 1)
        kaggle_preds.append(pred.item())

# 5. إنشاء ملف التسليم
submission = pd.DataFrame({
    'ImageId': range(1, len(kaggle_preds) + 1),
    'Label': kaggle_preds
})

submission.to_csv('submission.csv', index=False)
print(f"✅ تم حفظ ملف التسليم: submission.csv ({len(kaggle_preds)} توقع)")

# 6. حفظ النموذج للاستخدام المستقبلي
torch.save({
    'model_state_dict': model.state_dict(),
    'train_mean': train_mean,
    'train_std': train_std,
    'best_val_acc': best_val_acc,
}, 'digit_cnn_final.pth')
print("💾 تم حفظ النموذج: digit_cnn_final.pth")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# أعد قراءة ملف التسليم
sub = pd.read_csv('submission.csv')

# باقي الكود كما هو
test_df = pd.read_csv('/kaggle/input/competitions/digit-recognizer/test.csv')
X_display = test_df.values[:10] / 255.0

plt.figure(figsize=(15, 3))
for i in range(10):
    plt.subplot(1, 10, i+1)
    plt.imshow(X_display[i].reshape(28, 28), cmap='gray')
    plt.title(f"Pred: {sub.iloc[i]['Label']}")
    plt.axis('off')
plt.show()